# AIC 2026 — merge 873 InfoShot++ video shards

Notebook này chỉ hợp nhất **metadata/index** của các extraction session. Nó không giải
nén và không copy lại khoảng 200 GB keyframe, nên phù hợp với giới hạn output của
Kaggle.

```text
attached notebook outputs
  ├─ session A / archives/*.keyframes.zip + registry/*.csv
  ├─ session B / archives/*.keyframes.zip + registry/*.csv
  └─ ...
            ↓ audit cứng
      đủ 873 video, L21..L30, không duplicate,
      session hoàn chỉnh, pipeline signature đồng nhất
            ↓ metadata merge
      frame_registry.parquet
      asset_index.csv
      run_summary.csv
      source_sessions.csv
      merge_manifest.json
            ↓ tùy chọn
      publish index nhỏ thành Kaggle Dataset
```

Các output version cần merge của `lhuyhongphm`, `baovo123`, `vnhtbo` và `baovods` đã
được khai báo rõ trong `notebook_versions_by_handle`. Notebook dùng KaggleHub resolve
từng version, sau đó tự tìm mọi extraction session bên trong output đó. `input_roots`
để rỗng nhằm tránh attach lại latest version và tạo duplicate.

Dataset index sau khi publish **không chứa JPEG/ZIP keyframe**. Khi sử dụng index ở
notebook downstream, vẫn phải attach các notebook-output shard nguồn để các đường dẫn
asset có hiệu lực.


## 1. Cấu hình

Mặc định audit đúng corpus AIC 2025 S1: `L21..L30`, tổng cộng 873 video. Không bật GPU
vì merge chỉ đọc CSV/JSON và ghi Parquet.


In [ ]:
CONFIG = {
    # Để rỗng vì các version cụ thể bên dưới sẽ được resolve qua KaggleHub.
    # Không direct-mount latest song song, nếu không video ở latest sẽ bị duplicate.
    "input_roots": [],
    # Fallback cho historical outputs không hiện sẵn dưới input_roots. Giá trị là
    # số Version trên Kaggle, không phải title/name hay scriptVersionId.
    # Chỉ điền khi version đó chưa được direct-mount, nếu không sẽ duplicate.
    "notebook_versions_by_handle": {
        "lhuyhongphm/keyframe-final": [1, 2],
        "baovo123/keyframe-final": [1, 2, 3, 4],
        "vnhtbo/keyframe-final": [1, 2, 3, 4, 5],
        "baovods/keyframe-final": [1, 2, 3, 4, 5],
    },
    "output_root": "/kaggle/working/aic2026_infoshootpp_merged_index",

    "expected_categories": [f"L{i:02d}" for i in range(21, 31)],
    "expected_total_videos": 873,
    "enforce_expected_total": True,
    "strict_session_complete": True,
    # Chỉ yêu cầu các shard giống nhau; notebook merge không tự tạo signature mới.
    "strict_pipeline_signature": True,
    "validate_asset_exists": True,

    # Luôn giữ False cho bộ keyframe ~200 GB. Notebook chỉ tạo metadata index.
    "copy_assets": False,
    "write_jsonl": False,              # Parquet nhỏ và nhanh hơn JSONL.
    "write_per_video_registry": False, # Global Parquet đã chứa toàn bộ frame.
    "copy_map_keyframes": False,       # Có thể suy ra map từ global registry.
    "clean_output_on_rerun": True,

    # Publish chỉ dataset INDEX nhỏ; không upload lại keyframe assets.
    "publish_dataset": False,
    "dataset_handle": "vnhtbo/aic2026-infoshootpp-merged-index",
    "dataset_version_notes": "Merged metadata index for 873 AIC videos",
}

assert CONFIG["copy_assets"] is False, "Notebook này cố ý không copy 200 GB assets"


## 2. Imports và helpers


In [ ]:
from __future__ import annotations

import gc
import json
import os
import re
import shutil
import time
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


CANONICAL_VIDEO_RE = re.compile(r"(?i)^([KL]\d{2})_V(\d+)$")
HEAVY_DIR_NAMES = {
    "archives", "keyframes", "selector-embeddings", "final-embeddings",
    "cache", ".cache", "__pycache__",
}


def resolve_input_roots():
    roots = [Path(raw) for raw in CONFIG["input_roots"]]
    version_map = CONFIG.get("notebook_versions_by_handle", {}) or {}
    if any(versions for versions in version_map.values()):
        import kagglehub

        for raw_handle, raw_versions in version_map.items():
            handle = str(raw_handle).strip().strip("/")
            for version in sorted({int(value) for value in raw_versions}):
                version_handle = f"{handle}/versions/{version}"
                print("Resolve notebook output:", version_handle)
                resolved = kagglehub.notebook_output_download(version_handle)
                roots.append(Path(resolved))
    unique = []
    seen = set()
    for root in roots:
        key = str(root.resolve()) if root.exists() else str(root)
        if key not in seen:
            seen.add(key)
            unique.append(root)
    return unique


RESOLVED_INPUT_ROOTS = resolve_input_roots()


def atomic_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2, default=str),
        encoding="utf-8",
    )
    os.replace(tmp, path)


def natural_video_key(value):
    match = CANONICAL_VIDEO_RE.fullmatch(str(value))
    if not match:
        return (str(value), -1)
    return (match.group(1).upper(), int(match.group(2)))


def human_bytes(value):
    value = float(value)
    for unit in ["B", "KiB", "MiB", "GiB", "TiB"]:
        if value < 1024 or unit == "TiB":
            return f"{value:.2f} {unit}"
        value /= 1024


def input_locator(session_root):
    session_root = Path(session_root).resolve()
    for raw in RESOLVED_INPUT_ROOTS:
        input_root = Path(raw).resolve()
        try:
            relative = session_root.relative_to(input_root)
            return str(input_root), relative.as_posix()
        except ValueError:
            continue
    return None, None


## 3. Discover session nhanh

Cell này chỉ tìm `run_manifest.json`. Khi duyệt, nó chủ động bỏ qua `archives/`,
`keyframes/` và embedding folders nên không đi qua hàng trăm GB asset.


In [ ]:
def discover_session_roots(input_roots):
    if isinstance(input_roots, (str, Path)):
        input_roots = [input_roots]
    manifests = []
    for raw in input_roots:
        root = Path(raw)
        if not root.exists():
            raise FileNotFoundError(f"Input root không tồn tại: {root}")
        direct = root / "run_manifest.json"
        if direct.exists():
            manifests.append(direct)
            continue
        for walk_root, dir_names, file_names in os.walk(root):
            dir_names[:] = [
                name for name in dir_names
                if name.casefold() not in HEAVY_DIR_NAMES
            ]
            if "run_manifest.json" in file_names:
                manifests.append(Path(walk_root) / "run_manifest.json")

    unique = []
    seen = set()
    for manifest in sorted(manifests, key=lambda path: str(path)):
        resolved = str(manifest.parent.resolve())
        if resolved not in seen:
            seen.add(resolved)
            unique.append(manifest.parent)
    if not unique:
        raise FileNotFoundError(
            "Không tìm thấy run_manifest.json. Hãy attach full notebook output, "
            "không chỉ registry ZIP."
        )
    return unique


SESSION_ROOTS = discover_session_roots(RESOLVED_INPUT_ROOTS)
print(f"Discovered {len(SESSION_ROOTS)} extraction session(s):")
for root in SESSION_ROOTS:
    print(" -", root)


## 4. Audit bắt buộc trước merge

Audit kiểm tra session state, số registry, canonical ID, category, duplicate video và
pipeline signature. Nó chỉ đọc hai cột `video_id/category` ở vòng audit; keyframe ZIP
không bị mở hay CRC lại.


In [ ]:
def audit_sources(session_roots):
    session_rows = []
    entries = {}
    duplicate_rows = []
    problems = []
    signatures = set()

    for root in session_roots:
        manifest_path = root / "run_manifest.json"
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        if manifest.get("manifest_type", "session") != "session":
            problems.append(f"Không phải extraction session: {root}")
            continue

        session_id = str(manifest.get("session_id") or root.name)
        signature = manifest.get("pipeline_signature")
        signatures.add(signature)
        registry_files = [
            path for path in sorted((root / "registry").glob("*.csv"))
            if "__smoke" not in path.stem
        ]
        expected_session = int(manifest.get("mergeable_video_count", -1))
        completed_session = int(manifest.get("completed_videos", -1))
        session_complete = bool(manifest.get("session_complete", False))

        if CONFIG["strict_session_complete"] and (
            not session_complete
            or expected_session < 0
            or len(registry_files) != expected_session
            or completed_session != expected_session
        ):
            problems.append(
                f"Session chưa hoàn chỉnh {session_id}: complete={session_complete}, "
                f"mergeable={expected_session}, registry={len(registry_files)}, "
                f"completed={completed_session}"
            )

        video_ids_in_session = []
        categories_in_session = set()
        for registry_path in registry_files:
            try:
                identity = pd.read_csv(
                    registry_path,
                    usecols=lambda col: col in {"video_id", "category"},
                ).drop_duplicates()
            except Exception as exc:
                problems.append(f"Không đọc được {registry_path}: {exc}")
                continue
            if set(identity.columns) != {"video_id", "category"} or len(identity) != 1:
                problems.append(
                    f"Registry phải chứa đúng một video/category: {registry_path}"
                )
                continue
            video_id = str(identity.iloc[0]["video_id"])
            category = str(identity.iloc[0]["category"]).upper()
            match = CANONICAL_VIDEO_RE.fullmatch(video_id)
            if not match or match.group(1).upper() != category:
                problems.append(f"ID/category không canonical: {video_id} / {category}")
                continue
            if video_id in entries:
                duplicate_rows.append({
                    "video_id": video_id,
                    "first_session": entries[video_id]["session_id"],
                    "second_session": session_id,
                    "first_root": str(entries[video_id]["root"]),
                    "second_root": str(root),
                })
                continue
            input_root, session_relpath = input_locator(root)
            entries[video_id] = {
                "video_id": video_id,
                "category": category,
                "session_id": session_id,
                "root": root,
                "input_root": input_root,
                "session_relpath": session_relpath,
                "registry_path": registry_path,
                "manifest": manifest,
            }
            video_ids_in_session.append(video_id)
            categories_in_session.add(category)

        session_rows.append({
            "session_id": session_id,
            "root": str(root),
            "categories": ",".join(sorted(categories_in_session)),
            "videos_found": len(video_ids_in_session),
            "manifest_mergeable": expected_session,
            "manifest_completed": completed_session,
            "session_complete": session_complete,
            "stopped_for_storage": bool(manifest.get("stopped_for_storage", False)),
            "pipeline_signature": signature,
        })

    if duplicate_rows:
        problems.append(f"Có {len(duplicate_rows)} duplicate video_id giữa các session")

    expected_categories = set(CONFIG["expected_categories"])
    category_counts = {}
    for entry in entries.values():
        category_counts[entry["category"]] = category_counts.get(entry["category"], 0) + 1
    categories_found = set(category_counts)
    missing_categories = sorted(expected_categories - categories_found)
    unexpected_categories = sorted(categories_found - expected_categories)
    count_ok = len(entries) == int(CONFIG["expected_total_videos"])
    categories_ok = not missing_categories and not unexpected_categories
    signatures_ok = (
        not CONFIG["strict_pipeline_signature"]
        or (None not in signatures and len(signatures) == 1)
    )
    if not count_ok:
        problems.append(
            f"Tổng video {len(entries)} != expected {CONFIG['expected_total_videos']}"
        )
    if not categories_ok:
        problems.append(
            f"Category mismatch: missing={missing_categories}, unexpected={unexpected_categories}"
        )
    if not signatures_ok:
        problems.append(f"Pipeline signature không đồng nhất: {sorted(map(str, signatures))}")

    audit = {
        "created_at_unix": time.time(),
        "session_count": len(session_rows),
        "videos": len(entries),
        "expected_total_videos": int(CONFIG["expected_total_videos"]),
        "count_ok": count_ok,
        "expected_categories": sorted(expected_categories),
        "categories_found": sorted(categories_found),
        "category_counts": category_counts,
        "missing_categories": missing_categories,
        "unexpected_categories": unexpected_categories,
        "pipeline_signatures": sorted(map(str, signatures)),
        "signatures_ok": signatures_ok,
        "duplicate_video_count": len(duplicate_rows),
        "problems": problems,
        "audit_ok": not problems,
    }
    return pd.DataFrame(session_rows), entries, pd.DataFrame(duplicate_rows), audit


SESSION_AUDIT, VIDEO_ENTRIES, DUPLICATES, MERGE_AUDIT = audit_sources(SESSION_ROOTS)
display(SESSION_AUDIT)
display(
    pd.DataFrame(
        sorted(MERGE_AUDIT["category_counts"].items()),
        columns=["category", "videos"],
    )
)
if len(DUPLICATES):
    display(DUPLICATES)
print(json.dumps(MERGE_AUDIT, ensure_ascii=False, indent=2))

if CONFIG["enforce_expected_total"] and not MERGE_AUDIT["audit_ok"]:
    raise RuntimeError(
        "MERGE BLOCKED: source chưa đủ/sạch. Xem MERGE_AUDIT['problems']; "
        "không publish dataset khi chưa đúng 873 video."
    )


## 5. Merge metadata và remap asset URI

Với video đã nén, `image_path` mới có dạng:

```text
zip:///kaggle/input/.../archives/L25_V013.keyframes.zip!/keyframes/L25/L25_V013/....jpg
```

Với loose JPEG, đường dẫn được remap về session root đang mount. Các cột
`source_input_root` và `source_session_relpath` giúp remap lại nếu mount point thay đổi
ở notebook downstream.


In [ ]:
def prepare_output_root():
    destination = Path(CONFIG["output_root"])
    working = Path("/kaggle/working").resolve()
    resolved = destination.resolve()
    if resolved == working or working not in resolved.parents:
        raise ValueError("output_root bắt buộc là thư mục con cụ thể của /kaggle/working")
    if destination.exists():
        if not CONFIG["clean_output_on_rerun"]:
            raise FileExistsError(f"Output đã tồn tại: {destination}")
        shutil.rmtree(destination)
    destination.mkdir(parents=True)
    if CONFIG["write_per_video_registry"]:
        (destination / "registry").mkdir()
    if CONFIG["copy_map_keyframes"]:
        (destination / "map-keyframes").mkdir()
    return destination


def merge_metadata(entries, audit):
    if not audit["audit_ok"]:
        raise RuntimeError("Refuse merge: audit_ok=False")
    destination = prepare_output_root()
    frame_parts = []
    summary_rows = []
    asset_rows = []
    source_session_rows = []
    seen_sessions = set()

    ordered_ids = sorted(entries, key=natural_video_key)
    for position, video_id in enumerate(ordered_ids, 1):
        entry = entries[video_id]
        root = Path(entry["root"])
        category = entry["category"]
        session_id = entry["session_id"]
        df = pd.read_csv(entry["registry_path"])
        if df.empty:
            raise ValueError(f"Registry rỗng: {entry['registry_path']}")

        relative_image_dir = Path("keyframes") / category / video_id
        source_image_dir = root / relative_image_dir
        archive_rel_values = []
        if "image_archive_relpath" in df:
            archive_rel_values = (
                df["image_archive_relpath"].dropna().astype(str).unique().tolist()
            )
        if len(archive_rel_values) > 1:
            raise ValueError(f"{video_id}: registry trỏ tới nhiều archive")
        archive_relpath = Path(archive_rel_values[0]) if archive_rel_values else None
        archive_path = root / archive_relpath if archive_relpath else None
        has_archive = archive_path is not None

        if CONFIG["validate_asset_exists"]:
            if has_archive and not archive_path.is_file():
                raise FileNotFoundError(f"Thiếu archive: {archive_path}")
            if not has_archive and not source_image_dir.is_dir():
                raise FileNotFoundError(
                    f"Thiếu cả archive lẫn loose keyframe directory: {source_image_dir}"
                )

        asset_relpaths = []
        image_paths = []
        for row in df.itertuples(index=False):
            old_image_path = getattr(row, "image_path", None)
            if old_image_path is None or pd.isna(old_image_path):
                asset_relpaths.append(None)
                image_paths.append(None)
                continue
            archive_member = getattr(row, "image_archive_member", None)
            if has_archive and archive_member is not None and not pd.isna(archive_member):
                member = Path(str(archive_member)).as_posix()
                asset_relpaths.append(member)
                image_paths.append(f"zip://{archive_path}!/{member}")
            else:
                relative_path = relative_image_dir / Path(str(old_image_path)).name
                asset_relpaths.append(relative_path.as_posix())
                image_paths.append(str(root / relative_path))

        df["source_session_id"] = session_id
        df["source_input_root"] = entry["input_root"]
        df["source_session_relpath"] = entry["session_relpath"]
        df["source_session_root"] = str(root)
        df["asset_storage"] = "zip" if has_archive else "loose"
        df["asset_relpath"] = asset_relpaths
        if has_archive:
            df["image_archive_relpath"] = archive_relpath.as_posix()
            df["image_archive_path"] = str(archive_path)
        if "image_path" in df:
            df["image_path"] = image_paths
        frame_parts.append(df)

        if CONFIG["write_per_video_registry"]:
            df.to_parquet(destination / "registry" / f"{video_id}.parquet", index=False)
        if CONFIG["copy_map_keyframes"]:
            source_map = root / "map-keyframes" / f"{video_id}.csv"
            if source_map.exists():
                shutil.copy2(source_map, destination / "map-keyframes" / source_map.name)

        stats_path = root / "registry" / f"{video_id}.stats.json"
        if stats_path.exists():
            stats = json.loads(stats_path.read_text(encoding="utf-8"))
            stats["source_session_id"] = session_id
            summary_rows.append(stats)

        archive_bytes = (
            archive_path.stat().st_size
            if has_archive and archive_path.exists() else None
        )
        asset_rows.append({
            "video_id": video_id,
            "category": category,
            "keyframe_count": int(len(df)),
            "asset_storage": "zip" if has_archive else "loose",
            "source_session_id": session_id,
            "source_input_root": entry["input_root"],
            "source_session_relpath": entry["session_relpath"],
            "source_session_root": str(root),
            "keyframe_dir": str(source_image_dir),
            "keyframe_archive_relpath": archive_relpath.as_posix() if has_archive else None,
            "keyframe_archive_path": str(archive_path) if has_archive else None,
            "archive_bytes": archive_bytes,
        })

        if session_id not in seen_sessions:
            seen_sessions.add(session_id)
            manifest = entry["manifest"]
            source_session_rows.append({
                "session_id": session_id,
                "source_input_root": entry["input_root"],
                "source_session_relpath": entry["session_relpath"],
                "source_session_root": str(root),
                "pipeline_signature": manifest.get("pipeline_signature"),
                "mergeable_video_count": manifest.get("mergeable_video_count"),
                "stopped_for_storage": manifest.get("stopped_for_storage"),
                "next_range_start": manifest.get("next_range_start"),
                "next_video_id": manifest.get("next_video_id"),
            })

        if position % 25 == 0 or position == len(ordered_ids):
            print(f"Prepared {position:,}/{len(ordered_ids):,} video registries")

    registry = pd.concat(frame_parts, ignore_index=True, sort=False)
    registry["_video_number"] = registry["video_id"].astype(str).str.extract(
        r"_V(\d+)$", expand=False
    ).astype("int64")
    registry = registry.sort_values(
        ["category", "_video_number", "frame_idx"], kind="stable"
    ).drop(columns="_video_number").reset_index(drop=True)
    duplicate_frames = registry.duplicated(["video_id", "frame_idx"])
    if duplicate_frames.any():
        sample = registry.loc[duplicate_frames, ["video_id", "frame_idx"]].head()
        raise RuntimeError(f"Duplicate (video_id, frame_idx):\n{sample}")

    registry.to_parquet(
        destination / "frame_registry.parquet",
        index=False,
        compression="zstd",
    )
    if CONFIG["write_jsonl"]:
        registry.to_json(
            destination / "frame_registry.jsonl",
            orient="records",
            lines=True,
            force_ascii=False,
        )
    summary = pd.DataFrame(summary_rows)
    if len(summary):
        summary.to_csv(destination / "run_summary.csv", index=False)
    asset_index = pd.DataFrame(asset_rows).sort_values(
        ["category", "video_id"], kind="stable"
    )
    asset_index.to_csv(destination / "asset_index.csv", index=False)
    pd.DataFrame(source_session_rows).to_csv(
        destination / "source_sessions.csv", index=False
    )
    atomic_json(destination / "merge_audit.json", audit)

    output_files = [path for path in destination.rglob("*") if path.is_file()]
    manifest = {
        "manifest_type": "infoshootpp-metadata-merge",
        "created_at_unix": time.time(),
        "copy_assets": False,
        "expected_total_videos": CONFIG["expected_total_videos"],
        "merged_videos": int(registry["video_id"].nunique()),
        "merged_keyframes": int(len(registry)),
        "source_session_count": len(source_session_rows),
        "pipeline_signatures": audit["pipeline_signatures"],
        "audit_ok": audit["audit_ok"],
        "output_root": str(destination),
        "note": "Assets remain in attached source notebook outputs.",
    }
    atomic_json(destination / "merge_manifest.json", manifest)
    output_bytes = sum(path.stat().st_size for path in destination.rglob("*") if path.is_file())
    manifest["metadata_output_bytes"] = int(output_bytes)
    atomic_json(destination / "merge_manifest.json", manifest)
    return destination, registry, summary, asset_index, manifest


MERGE_ROOT, FRAME_REGISTRY, RUN_SUMMARY, ASSET_INDEX, MERGE_MANIFEST = merge_metadata(
    VIDEO_ENTRIES, MERGE_AUDIT
)
gc.collect()
print(json.dumps(MERGE_MANIFEST, ensure_ascii=False, indent=2))


## 6. Diagnostics cuối

`merged_videos` phải là 873 và audit phải `true`. `archive_bytes` ở đây là dung lượng
asset nguồn được tham chiếu; nó không phải dung lượng output merge.


In [ ]:
category_summary = (
    FRAME_REGISTRY.groupby("category")
    .agg(videos=("video_id", "nunique"), keyframes=("frame_idx", "size"))
    .reset_index()
)
storage_summary = (
    ASSET_INDEX.groupby("asset_storage")
    .agg(videos=("video_id", "size"), source_bytes=("archive_bytes", "sum"))
    .reset_index()
)
storage_summary["source_size"] = storage_summary["source_bytes"].map(human_bytes)
display(category_summary)
display(storage_summary)

output_bytes = sum(path.stat().st_size for path in MERGE_ROOT.rglob("*") if path.is_file())
print("Metadata output:", human_bytes(output_bytes), "at", MERGE_ROOT)
print("Videos:", FRAME_REGISTRY.video_id.nunique())
print("Keyframes:", f"{len(FRAME_REGISTRY):,}")

assert FRAME_REGISTRY.video_id.nunique() == CONFIG["expected_total_videos"]
assert set(FRAME_REGISTRY.category.unique()) == set(CONFIG["expected_categories"])
assert not FRAME_REGISTRY.duplicated(["video_id", "frame_idx"]).any()


## 7. Tùy chọn publish index thành Kaggle Dataset

Đổi `publish_dataset=True` ở CONFIG rồi Run All. Cell này chỉ upload thư mục metadata
nhỏ trong `output_root`; không upload 200 GB ZIP/JPEG nguồn.

Nếu Dataset handle chưa tồn tại, KaggleHub sẽ tạo dataset; nếu đã tồn tại, nó tạo một
version dataset mới. Tài khoản chạy notebook cần có quyền ghi vào owner trong
`dataset_handle`.


In [ ]:
PUBLISHED_DATASET = None
if CONFIG["publish_dataset"]:
    if not MERGE_MANIFEST.get("audit_ok") or MERGE_MANIFEST.get("merged_videos") != 873:
        raise RuntimeError("Refuse publish: merge chưa audit đủ 873 video")
    import kagglehub

    PUBLISHED_DATASET = kagglehub.dataset_upload(
        CONFIG["dataset_handle"],
        str(MERGE_ROOT),
        version_notes=CONFIG["dataset_version_notes"],
    )
    print("Published:", PUBLISHED_DATASET or CONFIG["dataset_handle"])
else:
    print(
        "publish_dataset=False — metadata đã sẵn sàng trong", MERGE_ROOT,
        "\nBật True nếu muốn tạo/update Kaggle Dataset index."
    )


## 8. Cách đọc một ảnh từ registry đã merge

Ví dụ dưới đây chỉ đọc **một** frame để kiểm tra URI. Không giải nén toàn archive.


In [ ]:
def read_keyframe_bytes(row):
    import zipfile

    archive_path = row.get("image_archive_path")
    archive_member = row.get("image_archive_member")
    if pd.notna(archive_path) and pd.notna(archive_member):
        with zipfile.ZipFile(str(archive_path), "r") as archive:
            return archive.read(str(archive_member))
    image_path = row.get("image_path")
    if image_path is None or pd.isna(image_path):
        raise FileNotFoundError("Registry row không có image asset")
    return Path(str(image_path)).read_bytes()


print("Helper read_keyframe_bytes(row) đã sẵn sàng.")
